# Gamil RAG project

### Step 1: Import library

In [16]:
import os
import ollama
from openai import OpenAI
import json
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
import gradio as gr
from gmail.gmail_function import login_gmail, get_emails_list
from tqdm import tqdm
from IPython.display import display, Markdown
from pydantic import BaseModel, Field

### Step 2: setting variables and environment

In [8]:
load_dotenv(override = True)
ollama_api_key = os.getenv("OLLAMA_API_KEY")
hf_token = os.getenv("HF_TOKEN")
credential_path = r'.credentials/credentials.json' # create credentials from Gmail API
token_path = r'.credentials/token.json' # Token will be created the first time of login to be used for subsequent login
login(hf_token, add_to_git_credential=True)

# # setup ollama
llm_client = OpenAI(base_url = "http://localhost:11434/v1/", api_key = ollama_api_key)
llm_model = "gemma3:1b"

# setup lms
# llm_client = OpenAI(base_url = "http://localhost:1234/v1/", api_key = "lms")
# llm_model = "google/gemma-3-4b"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Step 3: Check if gmail credentials exist

In [3]:
if not os.path.exists(credential_path):
    print(f"Error: Credentials file '{credential_path}' does not exist.")
else:
    print("Credential file exists!")

Credential file exists!


### Step 4: Login gmail with credentials or token

In [4]:
service = login_gmail(credential_path, token_path)

### Step 5: Retrieve Gmail for the past 7 days

In [5]:
emails = get_emails_list(service, days=3)

In [12]:
print(len(emails))

43


### Step 6: Create a summary for every gmail

In [ ]:
class Result(BaseModel):
    summary: str = Field(description = "email summary")
    metadata: dict

class Email(BaseModel):
    title: str = Field(description = "Email title")
    body: str = Field(description = "Email body")
    summary: str = Field(description = "Email summary based on title and body")
    sender: str = Field(description = "email sender details")
    date_receival: str = Field(description = "Date of email received")

    def as_result(self):
        metadata = {"sender": self.sender, "date_receival": self.date_receival}
        return Result(summary = self.summary, metadata = metadata)

class Emails(BaseModel):
    email: list[Email]

In [ ]:
system_prompt = f"""
You will be given the title and body of an email, help to create a summary for it.
Some emails might not have any plain text for its body.
"""

def preprocess_email(emails):
  for email in tqdm(emails):
    title = email["title"]
    body = email["body"]
    user_prompt = f""" 
    Please help to provide summary of the email based on title and/or body with one or several sentences. 
    Keep it clear and concise, straight to the point. 
    No need to ask any further questions.

    Here is the title of the email:
    f{title}

    Here is the body of the email:
    f{body if body else "This email body does not contain any text. Please summarize based on the title only."}
    """
    messages = [{"role":"system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

    response = llm_client.chat.completions.create(model = llm_model, messages = messages, max_tokens = 2000)

    email["summary"] = response.choices[0].message.content
  
  return emails

100%|██████████| 43/43 [01:25<00:00,  1.99s/it]


In [15]:
for email in emails:
    display(Markdown(email["summary"]))



Please summarize the email titled “fGrab receipt” based solely on the provided title.

This email alerts you to potential fTransaction Alerts, prompting you to review your account for any suspicious activity.

This email is a notification about a Grab receipt, likely regarding a recent purchase.

Interactive Brokers notified Broadcom Inc. of a cash dividend.

Okay, here's a breakdown of the information provided in the text:

**Summary of the Changi Rewards Account Statement**

The text presents a Changi Rewards account statement, detailing the following:

*   **Points Status:** The user has a relatively high point balance and is close to expiry.
*   **Expiry Date:** Their points are expiring on [Date].
*   **Points Value:** The total points available is [Amount].
*   **Points Usage:** The user has used [Amount] of their points.
*   **Expiry Reason:**  The points are expiring due to the expiry date.

**Additional Details:**

*   **Link to More Information:**  There is a link to a website offering more details about the account, including the privacy policy and other relevant information (like the points limit).
*   **Disclaimer:** Copyright information.

**In essence, the text is providing a notification that the user's rewards points are expiring soon.**

Do you have any specific questions about this information, or would you like me to summarize it differently? For example, are you interested in:

*   How to handle the expiring points?
*   How to manage your Changi Rewards account?

...

Okay, here's a summary of the information provided:

**You've received an email from LinkedIn regarding a job alert.** The email is designed for reliability engineer Jun Ming Soh.

**Key Points:**

*   **Job Alert:** The email is a job alert.
*   **Recruiter:** The email is from Reliability Engineer Jun Ming Soh.
*   **Background:** Jun Ming Soh is responsible for Machine-Learning and Data Analytics.
*   **Email Context:** Jun Ming Soh is receiving a job alert.
*   **Link to Help:** The email includes a link to LinkedIn’s help center.
*   **Link to Job:** The email also includes a link to the job listing itself.

Let me know if you’d like me to do anything else with this information, such as:

*   Summarize the email content
*   Extract specific details
*   Answer a question you have about the email

The email confirms a Pilates session with Aman G at Heartbeat, starting on March 23, 2026, from 7:45 PM to 8:45 PM.  Details can be viewed at https://activesg.gov.sg/programmes/confirmation/LDVYajQBPhhDARjDTlgwh.

You have received a transfer notification from fdigibank.

This email alerts you that a successful bill payment has been made to your account.

Here’s a summary of the email:

The email warns readers about potential scams related to the FIFA World Cup 2026, emphasizing the risk of individuals being targeted with fake ticket sales, website fraud, and other deceptive schemes. It advises users to be vigilant and take precautions against financial loss.

In [14]:
display(Markdown(response.choices[0].message.content))

Okay, here’s a breakdown of your Gmail content, presented in a Markdown format, prioritizing key information:

**Key Findings & Opportunities:**

*   **Job Alerts:** You've received numerous job alerts primarily focused on AI/ML Engineering roles (Software Engineer, Machine Learning Engineer, Research Scientist) at companies like Bosch, Apple, Micron, NVIDIA, and Meta.
*   **Potential Security Alert:**  You’ve received a security alert from Google indicating that `gmail_rag_project` has access to some of your account data. You should review your Google Account activity and secure it immediately.
*   **Job Offer - Potential:** A job alert for a Senior Mechanical Engineer position at Pfizer was also detected.

**Detailed Breakdown by Email Type:**

*   **Job Alerts (LinkedIn):**  You're being actively notified about AI/ML Engineering positions across various companies, including Apple, Meta, NVIDIA, and Micron. These alerts are driving the majority of your email traffic.
*   **Security Alert (Google):** This is a critical alert indicating unauthorized access to your Google Account. You *must* investigate this immediately by checking account activity and reviewing permissions.
*   **Marketing Emails:**  You've received promotional emails from:
    *   CTOS (Financial Services) - Offering credit report services.
    *   PayLah! (Mobile Payment Platform) – Alerts about transactions and a reminder of phishing scams.
    *   Webull (Brokerage Platform) –  Promoting their services and providing links to their website, app, and social media channels.
*   **Other Emails:**
    *   A reminder about Phishing emails from Webull.
    *   A hospitalization claim email from AIA Singapore.

**Important Note:** The `gmail_rag_project` security alert is the highest priority.  Take immediate action to secure your Google Account.

Do you want me to delve deeper into any specific aspect of this information, such as:

*   Analyzing the job alerts in more detail (e.g., filtering by company or role)?
*   Providing instructions on how to investigate the security alert?

### Step 8: Create a chatbot for user to ask further questions

In [15]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] +\
                [{"role":h["role"], "content": h["content"]} for h in history]+\
                    [{"role": "user", "content": message}]
    response = llm_client.chat.completions.create(model = llm_model, messages = messages, max_tokens = 2000)
    return response.choices[0].message.content

In [16]:
with gr.Blocks() as ui:
    with gr.Row():
        gr.ChatInterface(fn=chat)
        
ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
